# Data Preparation

Der erste Schritt normalisiert die ROhdaten und reichert sie mit query-relevanten Daten an. Das Anreichern wird mit einem LLM durchgeführt. Die Daten prüfe ich nur stichprobenartig, sie gelten erstmal als so wahr.

- Rohdaten: Aus einem Vibecoding-Projekt.
- LLM: Mistral 

Wegen mehrmaliger Crashes werden Beschreibungen und technische Daten erweitern jeweils die Rohdaten erweitern und ihre Ergebnisse zwischenspeichern, ehe sie zuletzt zusammegeführt werden. Es werden nur Rohdaten verwendet, die ausreichend Beschreibung UND technische Daten liefert.

In [56]:
import os
import json
import pandas as pd
from tqdm import tqdm
from mistralai import Mistral
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv('MISTRAL_API_KEY')
model = 'mistral-medium-2508'
client = Mistral(api_key=api_key, timeout_ms=120000)

def agent_request(system_promt, schema, content):
            
    response = client.chat.complete(
        model = model,
        messages = [
            {
                'role': 'system',
                'content': system_promt
            },
            {
                'role': 'user',
                'content': content,
            }
        ],
        response_format = {
            "type": "json_object",
            "json_schema": schema
        }
    )

    return response

with open('../data/raw/products_raw.json', 'r') as f:
     products_raw = json.load(f)

products_filtered = [
    p for p in products_raw
    if len(p.get('description', '')) >= 100 and len(p.get('specs', [])) != 0
]

products_filtered = products_filtered[:30]

## Beschreibungen

Die Beschreibungen sollen so gegliedert sein, dass jeder Absatz ein Thema behandelt und das Produktname und Hersteller genannt wird. Werbliche Texte sollen entfernt werden. Die Antwort wird als JSON erwartet, es wird die dazu _response_format_ der API genutzt.

In [58]:
products_with_descs = products_filtered

with open('../data/promts/descs_agent.md', 'r') as f:
    descs_promt = f.read()

with open('../data/promts/descs_schema.json', 'r')as f:
    descs_schema = json.load(f)

for product in tqdm(products_with_descs, total=len(products_with_descs)):

    desc_response = agent_request(descs_promt, descs_schema, product['description'])
    product['desc_documents'] = json.loads(desc_response.choices[0].message.content)
    product['desc_usage'] = desc_response.usage.model_dump()

with open("../data/processed/products_w_beschreibung.json", "w", encoding="utf-8") as f:
    json.dump(products_with_descs, f, ensure_ascii=False, indent=2)

100%|██████████| 30/30 [06:28<00:00, 12.94s/it]


## Technische Daten

Bei den technischen Daten werden prinzipiell die selben Daten wie zur Beschreibung hinzugefügt, allerdings pro Objekt.

In [59]:
products_with_specs = products_filtered

with open('../data/promts/specs_agent.md', 'r') as f:
    specs_promt = f.read()

with open('../data/promts/specs_schema.json', 'r') as f:
    specs_schema = json.load(f)

for product in tqdm(products_with_specs, total=len(products_with_specs), desc="Products"):

    specs_serialaized = json.dumps(product['specs'])
    
    specs_response = agent_request(specs_promt, specs_schema, specs_serialaized)
    product['specs_documents'] = json.loads(specs_response.choices[0].message.content)
    product['specs_usage'] = specs_response.usage.model_dump()

with open("../data/processed/products_w_daten.json", "w", encoding="utf-8") as f:
    json.dump(products_with_specs, f, ensure_ascii=False, indent=2)

Products:   0%|          | 0/30 [00:00<?, ?it/s]

Products: 100%|██████████| 30/30 [13:25<00:00, 26.86s/it]


## Speichern

In [60]:
products_enriched = products_filtered

for i, (desc_prod, spec_prod) in enumerate(zip(products_with_descs, products_with_specs)):

    products_enriched[i]['specs_documents'] = spec_prod['specs_documents']
    products_enriched[i]['specs_usage'] = spec_prod['specs_usage']

    products_enriched[i]['desc_documents'] = desc_prod['desc_documents']
    products_enriched[i]['desc_usage'] = desc_prod['desc_usage']

with open("../data/processed/products_enriched.json", "w", encoding="utf-8") as f:
    json.dump(products_enriched, f, ensure_ascii=False, indent=2)

## Evaluation

Einmal nachsehen wie lange die Documents geworden sind und ob alle gefüllt wurden

In [61]:
with open('../data/processed/products_enriched.json', 'r', encoding='utf-8') as f:
    evaldata = json.load(f)

specs_chunks = []
descs_chunks = []
costs = []

for product in evaldata:

    costs.append({
        'id': product['id'],
        'type': 'descs',
        'prompt_tokens': product['desc_usage']['prompt_tokens'],
        'prompt_tokens': product['desc_usage']['completion_tokens'],
        'prompt_tokens': product['desc_usage']['total_tokens'],
    })

    costs.append({
        'id': product['id'],
        'type': 'specs',
        'prompt_tokens': product['specs_usage']['prompt_tokens'],
        'prompt_tokens': product['specs_usage']['completion_tokens'],
        'prompt_tokens': product['specs_usage']['total_tokens'],
    })

    for i, doc in enumerate(product.get('desc_documents')):

        descs_chunks.append({
            'id': product['id'],
            'num': f"{i:02d}",
            'doc': doc,
            'len': len(doc),
            'words': len(doc.split())
        })

    for i, spec in enumerate(product.get('specs_documents')):

        specs_chunks.append({
            'id': product['id'],
            'num': f"{i:02d}",
            'doc': spec['natural_language_description'],
            'len': len(spec['natural_language_description']),
            'words': len(spec['natural_language_description'].split())
        })

descs_df = pd.DataFrame(descs_chunks)
specs_df = pd.DataFrame(specs_chunks)
costs_df = pd.DataFrame(costs)

# print(specs_df.info())
# print(specs_df.head(5))
# print(specs_df.columns)

In [62]:
# Prüfen wieviele Absätze pro Produkt (min, max, mean)
# Prüfen wieviele Specs pro Produkt (min, max, mean)

print(f"Beschreibungen:\n{descs_df['words'].describe()}")
print("\n")
print(f"Technische Daten:\n{specs_df['words'].describe()}")

# Kurze und Lange Chunks anschauen
print(descs_df[descs_df['words'] <= 50].to_string())
print(specs_df[specs_df['words'] <= 5].to_string())

Beschreibungen:
count    152.000000
mean      91.171053
std       17.044071
min       51.000000
25%       80.000000
50%       90.000000
75%      102.000000
max      135.000000
Name: words, dtype: float64


Technische Daten:
count    1134.000000
mean       12.218695
std         4.013851
min         6.000000
25%         9.000000
50%        11.000000
75%        14.000000
max        33.000000
Name: words, dtype: float64
Empty DataFrame
Columns: [id, num, doc, len, words]
Index: []
Empty DataFrame
Columns: [id, num, doc, len, words]
Index: []


### Summary

Das Schema würde ich nochmal ändern. Die Kosten werde ich auch erstmal nur mitnehmen.

Anstelle die Länge zu zählen, habe ich die Worte zählen lassen um besser zu sehen wie viel Kontext im Chunk möglich ist. Alles in allem muss die Frage sitzen um genauer technische Daten zu erhalten. Ich könnte noch ein größeres Model nehmen um die Daten erstellen zu lassen? ...